# Clickbait Title/Body Transformer Training (Korean DeBERTa Base)
`title/body` split CSV를 사용해 `team-lucid/deberta-v3-base-korean` 계열의 DeBERTa를 파인튜닝합니다. 산출물은 별도 run으로 저장합니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil
from pathlib import Path


In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/sw_project/ai_features/clickbait_detection'
TRANSFORMER_SRC = os.path.join(DRIVE_ROOT, 'clickbait_transformer_finetune')
TRANSFORMER_WORK = '/content/clickbait_transformer_finetune'
DATA_SRC = os.path.join(DRIVE_ROOT, 'data')

assert os.path.exists(TRANSFORMER_SRC), f'Not found: {TRANSFORMER_SRC}'
assert os.path.exists(DATA_SRC), f'Not found: {DATA_SRC}'

if os.path.exists(TRANSFORMER_WORK):
    shutil.rmtree(TRANSFORMER_WORK)
shutil.copytree(TRANSFORMER_SRC, TRANSFORMER_WORK)
if os.path.exists(os.path.join(TRANSFORMER_WORK, 'data')):
    shutil.rmtree(os.path.join(TRANSFORMER_WORK, 'data'))
shutil.copytree(DATA_SRC, os.path.join(TRANSFORMER_WORK, 'data'), dirs_exist_ok=True)

os.chdir(TRANSFORMER_WORK)
!pip -q install -r requirements-colab.txt


In [ ]:
!python3 -u train_transformer.py \
  --model-name team-lucid/deberta-v3-base-korean \
  --train-data data/train.csv \
  --valid-data data/valid.csv \
  --test-data data/test.csv \
  --title-col title \
  --body-col body \
  --label-col label \
  --max-length 128 \
  --epochs 2 \
  --batch-size 4 \
  --learning-rate 1e-5 \
  --warmup-ratio 0.1 \
  --gradient-accumulation-steps 2 \
  --output-dir outputs/deberta_v3_base_title_body_run1 \
  --save-model-dir models/deberta_v3_base_title_body_run1


In [ ]:
!ls -lah models | sed -n '1,120p'


### Note
`team-lucid/deberta-v3-base-korean`는 `roberta-base`보다 무거울 수 있습니다. Colab에서 OOM이 나면 `--batch-size 2`로 낮추세요.
